# C15 — City Swap eval for Tier-1 combos (C11–C14)

City-swap robustness evaluation for the four Tier-1 combo challengers.

For scrubbing models (C11, C13) texts are scrubbed **after** city swap, matching training inference.

**Models**
- C11: `scrubbing_label_smoothing_eps01_2ep`
- C12: `label_smoothing_rdrop_eps01_alpha10_2ep`
- C13: `scrubbing_rdrop_alpha10_2ep`
- C14: `class_balanced_ls_eps01_beta099_2ep`

**Outputs**
- Summary CSV: `figures/challengers/c15_tier1_combo_city_swap_summary.csv`
- Per-model JSON: `notebooks/results/challenger_city_swap/c15_tier1_combo/<model_name>/`

In [1]:
import gc
import json
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

CWD = Path.cwd()
NOTEBOOKS_DIR = CWD.parent if CWD.name == "challengers" else CWD
PROJECT_ROOT = NOTEBOOKS_DIR.parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = NOTEBOOKS_DIR / "models" / "challengers"
FIGURES_DIR = PROJECT_ROOT / "figures" / "challengers"
RESULTS_DIR = NOTEBOOKS_DIR / "results" / "challenger_city_swap" / "c15_tier1_combo"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_CSV = FIGURES_DIR / "c15_tier1_combo_city_swap_summary.csv"
SWAP_CITIES = ["Москва", "Екатеринбург", "Новосибирск", "Краснодар", "Воронеж"]
BATCH_SIZE = 8
MAX_LENGTH = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"MODELS_DIR: {MODELS_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")
print(f"device: {device}")

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository
MODELS_DIR: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/models/challengers
RESULTS_DIR: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/challenger_city_swap/c15_tier1_combo
device: cpu


In [2]:
COMBO_MODELS = [
    {
        "notebook": "c11",
        "model_name": "scrubbing_label_smoothing_eps01_2ep",
        "method": "Scrub + LS + sqrt_rw",
        "scrub": True,
    },
    {
        "notebook": "c12",
        "model_name": "label_smoothing_rdrop_eps01_alpha10_2ep",
        "method": "LS + R-Drop + sqrt_rw",
        "scrub": False,
    },
    {
        "notebook": "c13",
        "model_name": "scrubbing_rdrop_alpha10_2ep",
        "method": "Scrub + R-Drop + sqrt_rw",
        "scrub": True,
    },
    {
        "notebook": "c14",
        "model_name": "class_balanced_ls_eps01_beta099_2ep",
        "method": "CBCE + LS + sqrt_rw",
        "scrub": False,
    },
]

resolved_rows = []
for item in COMBO_MODELS:
    model_dir = MODELS_DIR / item["model_name"]
    resolved_rows.append({
        "notebook": item["notebook"],
        "model_name": item["model_name"],
        "method": item["method"],
        "scrub": item["scrub"],
        "model_dir": str(model_dir),
        "exists": model_dir.exists() and (model_dir / "config.json").exists(),
    })

pd.DataFrame(resolved_rows)

,notebook,model_name,method,scrub,model_dir,exists
0,c11,scrubbing_label_smoothing_eps01_2ep,Scrub + LS + sqrt_rw,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True
1,c12,label_smoothing_rdrop_eps01_alpha10_2ep,LS + R-Drop + sqrt_rw,False,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True
2,c13,scrubbing_rdrop_alpha10_2ep,Scrub + R-Drop + sqrt_rw,True,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True
3,c14,class_balanced_ls_eps01_beta099_2ep,CBCE + LS + sqrt_rw,False,/Users/natashaagapova/Documents/A-INNOPOLIS/A-...,True


In [3]:
CITY_SWAP_PATTERNS = [
    "санкт-петербург", "нижний новгород", "ростов-на-дону",
    "набережные челны", "магнитогорск", "новосибирск",
    "екатеринбург", "красноярск", "волгоград", "калининград",
    "владивосток", "хабаровск", "ставрополь", "саратов",
    "челябинск", "самара", "казань", "москва", "омск",
    "воронеж", "пермь", "тюмень", "томск", "уфа",
    "тольятти", "барнаул", "иркутск", "пенза", "липецк",
    "кемерово", "сочи", "тверь", "минск", "алматы",
    "симферополь", "ярославль", "ульяновск", "ижевск",
    "оренбург", "мск", "спб", "питер",
]
CITY_SWAP_RE = re.compile(r"\b(" + "|".join(re.escape(c) for c in CITY_SWAP_PATTERNS) + r")\b", re.IGNORECASE)

SCRUB_WORDS = [
    "москва", "московская", "московский", "мск",
    "санкт-петербург", "петербург", "спб", "питер", "ленинград",
    "новосибирск", "екатеринбург", "казань", "нижний новгород",
    "челябинск", "самара", "омск", "ростов-на-дону", "уфа",
    "красноярск", "воронеж", "пермь", "волгоград",
    "краснодар", "саратов", "тюмень", "тольятти", "ижевск",
    "барнаул", "ульяновск", "иркутск", "хабаровск", "ярославль",
    "владивосток", "махачкала", "томск", "оренбург", "кемерово",
    "новокузнецк", "рязань", "астрахань", "пенза", "липецк",
    "калининград", "тула", "курск", "ставрополь", "сочи",
    "минск", "алматы", "киев", "симферополь",
    "область", "край", "республика", "регион",
    "забайкальский", "приморский", "краснодарский",
    "пенсионер", "пенсионерка", "пенсия", "пенсионный",
    "студент", "студентка", "выпускник", "выпускница",
    "молодой", "молодая", "junior", "senior",
]


def swap_cities_in_text(text, target_city):
    if pd.isna(text):
        return ""

    def replacer(match):
        orig = match.group(0)
        return target_city.capitalize() if orig and orig[0].isupper() else target_city.lower()

    return CITY_SWAP_RE.sub(replacer, str(text))


def scrub_text(text, mask_token="[MASK]"):
    if pd.isna(text):
        return ""
    result = str(text)
    for word in sorted(SCRUB_WORDS, key=len, reverse=True):
        result = re.compile(re.escape(word), re.IGNORECASE).sub(mask_token, result)
    return result


def prepare_texts(series, scrub=False, swap_city=None):
    texts = series.fillna("").astype(str)
    if swap_city is not None:
        texts = texts.apply(lambda x: swap_cities_in_text(x, swap_city))
    if scrub:
        texts = texts.apply(scrub_text)
    return texts.tolist()


def predict_batch(texts, model, tokenizer, batch_size=BATCH_SIZE):
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            outputs = model(**enc)
            preds = outputs.logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        del enc, outputs
    return np.array(all_preds)


def load_model_for_eval(model_dir):
    model_dir = Path(model_dir)
    if not (model_dir / "config.json").exists():
        return None
    tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
    model, loading_info = AutoModelForSequenceClassification.from_pretrained(
        str(model_dir), output_loading_info=True
    )
    missing = set(loading_info.get("missing_keys", []))
    if any(k.startswith("classifier.") for k in missing):
        return None
    le_path = model_dir / "label_encoder.joblib"
    if not le_path.exists():
        return None
    return {
        "model": model.to(device).eval(),
        "tokenizer": tokenizer,
        "label_encoder": joblib.load(le_path),
    }

In [4]:
df_test = pd.read_csv(DATA_DIR / "test.csv")
mapping = pd.read_csv(DATA_DIR / "label_to_supercategory_v1.csv")
label_to_super = dict(zip(mapping["label"], mapping["supercategory"]))

summary_rows = []

for item in COMBO_MODELS:
    model_dir = MODELS_DIR / item["model_name"]
    print("\n" + "=" * 80)
    print(f"[{item['notebook']}] {item['model_name']} ({item['method']})")

    row = {
        "notebook": item["notebook"],
        "model_name": item["model_name"],
        "method": item["method"],
        "scrub_at_inference": item["scrub"],
        "model_dir": str(model_dir.relative_to(PROJECT_ROOT)),
        "accuracy": None,
        "macro_f1": None,
        "overall_flip_rate": None,
        "status": "",
        "error": "",
    }

    if not model_dir.exists():
        row["status"] = "missing_model_dir"
        row["error"] = "Model directory not found"
        summary_rows.append(row)
        print("  Missing model directory")
        continue

    loaded = load_model_for_eval(model_dir)
    if loaded is None:
        row["status"] = "load_failed"
        row["error"] = "Could not load model/tokenizer/label encoder"
        summary_rows.append(row)
        print("  Load failed")
        continue

    model = loaded["model"]
    tokenizer = loaded["tokenizer"]
    le = loaded["label_encoder"]

    base_texts = prepare_texts(df_test["resume_text"], scrub=item["scrub"])
    orig_preds = predict_batch(base_texts, model, tokenizer)

    any_flip = np.zeros(len(df_test), dtype=bool)
    per_city = {}

    for swap_city in SWAP_CITIES:
        swapped_texts = prepare_texts(df_test["resume_text"], scrub=item["scrub"], swap_city=swap_city)
        swap_preds = predict_batch(swapped_texts, model, tokenizer)
        flipped = swap_preds != orig_preds
        flip_rate = float(flipped.mean())
        any_flip |= flipped
        per_city[swap_city] = flip_rate
        row[f"flip_{swap_city}"] = flip_rate
        print(f"  {swap_city}: {flip_rate:.3f} ({int(flipped.sum())}/{len(df_test)})")
        del swapped_texts, swap_preds, flipped
        gc.collect()

    y_true = le.transform(df_test["label"].map(label_to_super).fillna("generic_it_ops"))
    acc = float(accuracy_score(y_true, orig_preds))
    f1 = float(f1_score(y_true, orig_preds, average="macro"))
    overall_flip = float(any_flip.mean())

    row["accuracy"] = acc
    row["macro_f1"] = f1
    row["overall_flip_rate"] = overall_flip
    row["status"] = "ok"
    print(f"  Acc={acc:.3f}  F1={f1:.3f}  Flip={overall_flip:.3f}")

    out_dir = RESULTS_DIR / item["model_name"]
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "city_swap_summary.json").write_text(
        json.dumps(
            {
                "notebook": item["notebook"],
                "model_name": item["model_name"],
                "method": item["method"],
                "scrub_at_inference": item["scrub"],
                "accuracy": acc,
                "macro_f1": f1,
                "overall_flip_rate": overall_flip,
                "per_swap_city": {city: {"flip_rate": rate} for city, rate in per_city.items()},
            },
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    summary_rows.append(row)
    del model, tokenizer, le, orig_preds, any_flip
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False)
summary_df.to_csv(RESULTS_DIR / "city_swap_summary.csv", index=False)
summary_df


[c11] scrubbing_label_smoothing_eps01_2ep (Scrub + LS + sqrt_rw)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2487.75it/s, Materializing param=classifier.weight]                                      


  Москва: 0.002 (9/5510)
  Екатеринбург: 0.002 (9/5510)
  Новосибирск: 0.002 (9/5510)
  Краснодар: 0.002 (9/5510)
  Воронеж: 0.002 (9/5510)
  Acc=0.605  F1=0.620  Flip=0.002

[c12] label_smoothing_rdrop_eps01_alpha10_2ep (LS + R-Drop + sqrt_rw)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1683.80it/s, Materializing param=classifier.weight]                                      


  Москва: 0.017 (91/5510)
  Екатеринбург: 0.037 (204/5510)
  Новосибирск: 0.035 (193/5510)
  Краснодар: 0.032 (177/5510)
  Воронеж: 0.026 (141/5510)
  Acc=0.607  F1=0.620  Flip=0.060

[c13] scrubbing_rdrop_alpha10_2ep (Scrub + R-Drop + sqrt_rw)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2664.90it/s, Materializing param=classifier.weight]                                      


  Москва: 0.001 (5/5510)
  Екатеринбург: 0.001 (5/5510)
  Новосибирск: 0.001 (5/5510)
  Краснодар: 0.001 (5/5510)
  Воронеж: 0.001 (5/5510)
  Acc=0.606  F1=0.622  Flip=0.001

[c14] class_balanced_ls_eps01_beta099_2ep (CBCE + LS + sqrt_rw)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2696.75it/s, Materializing param=classifier.weight]                                      


  Москва: 0.032 (174/5510)
  Екатеринбург: 0.050 (278/5510)
  Новосибирск: 0.060 (329/5510)
  Краснодар: 0.046 (253/5510)
  Воронеж: 0.042 (233/5510)
  Acc=0.603  F1=0.617  Flip=0.086


,notebook,model_name,method,scrub_at_inference,model_dir,accuracy,macro_f1,overall_flip_rate,status,error,flip_Москва,flip_Екатеринбург,flip_Новосибирск,flip_Краснодар,flip_Воронеж
0,c11,scrubbing_label_smoothing_eps01_2ep,Scrub + LS + sqrt_rw,True,notebooks/models/challengers/scrubbing_label_s...,0.605445,0.620370,0.001633,ok,,0.001633,0.001633,0.001633,0.001633,0.001633
1,c12,label_smoothing_rdrop_eps01_alpha10_2ep,LS + R-Drop + sqrt_rw,False,notebooks/models/challengers/label_smoothing_r...,0.606897,0.620384,0.060436,ok,,0.016515,0.037024,0.035027,0.032123,0.025590
2,c13,scrubbing_rdrop_alpha10_2ep,Scrub + R-Drop + sqrt_rw,True,notebooks/models/challengers/scrubbing_rdrop_a...,0.606352,0.622049,0.000907,ok,,0.000907,0.000907,0.000907,0.000907,0.000907
3,c14,class_balanced_ls_eps01_beta099_2ep,CBCE + LS + sqrt_rw,False,notebooks/models/challengers/class_balanced_ls...,0.603085,0.617245,0.086207,ok,,0.031579,0.050454,0.059710,0.045917,0.042287


In [5]:
lines = ["=== C15 Tier-1 combo city-swap summary ==="]
for _, row in summary_df.iterrows():
    lines.extend([
        f"notebook: {row['notebook']}",
        f"model_name: {row['model_name']}",
        f"status: {row['status']}",
        f"accuracy: {row['accuracy']}",
        f"macro_f1: {row['macro_f1']}",
        f"overall_flip_rate: {row['overall_flip_rate']}",
        f"error: {row['error'] if row['error'] else '-'}",
        "-" * 60,
    ])

report_text = "\n".join(lines)
(RESULTS_DIR / "city_swap_report.txt").write_text(report_text, encoding="utf-8")
print(report_text)
print(f"\nSaved: {SUMMARY_CSV}")

=== C15 Tier-1 combo city-swap summary ===
notebook: c11
model_name: scrubbing_label_smoothing_eps01_2ep
status: ok
accuracy: 0.6054446460980036
macro_f1: 0.6203699309632715
overall_flip_rate: 0.001633393829401089
error: -
------------------------------------------------------------
notebook: c12
model_name: label_smoothing_rdrop_eps01_alpha10_2ep
status: ok
accuracy: 0.6068965517241379
macro_f1: 0.6203841850650377
overall_flip_rate: 0.06043557168784029
error: -
------------------------------------------------------------
notebook: c13
model_name: scrubbing_rdrop_alpha10_2ep
status: ok
accuracy: 0.6063520871143375
macro_f1: 0.622049020599317
overall_flip_rate: 0.0009074410163339383
error: -
------------------------------------------------------------
notebook: c14
model_name: class_balanced_ls_eps01_beta099_2ep
status: ok
accuracy: 0.6030852994555354
macro_f1: 0.6172445794962903
overall_flip_rate: 0.08620689655172414
error: -
------------------------------------------------------------